# 06 CatBoost Plus Graph
Этап 12: transition graph и гибридный reranking.

In [ ]:
import json
from pathlib import Path
import importlib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

from _shared_notebook_utils import RESEARCH_CHECKPOINT_DIR

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
registry_path = ROOT / "artifacts" / "model_registry.json"
with open(registry_path, "r", encoding="utf-8") as f:
    registry = json.load(f)

GRAPH_MODEL_KEY = "baseline"  # baseline|improved; graph reranking usually starts from baseline proba.
model_entry = registry["models"][GRAPH_MODEL_KEY]
model_path = ROOT / model_entry["model_path"]
meta_path = ROOT / model_entry["meta_path"]

if not model_path.exists() or not meta_path.exists():
    raise FileNotFoundError(f"Model artifacts not found for {GRAPH_MODEL_KEY}: {model_path} / {meta_path}")

baseline_dir = RESEARCH_CHECKPOINT_DIR / "baseline"
train_df = pd.read_pickle(baseline_dir / "train_df.pkl")
val_df = pd.read_pickle(baseline_dir / "val_df.pkl")
test_df = pd.read_pickle(baseline_dir / "test_df.pkl")

with open(meta_path, "r", encoding="utf-8") as f:
    model_meta = json.load(f)

feature_columns = model_meta.get("feature_columns", model_meta.get("feature_cols", []))
if not feature_columns:
    raise ValueError("feature columns are missing in model meta")

if "id_to_target" in model_meta:
    id_to_target = {int(k): str(v) for k, v in model_meta["id_to_target"].items()}
    class_names = [id_to_target[i] for i in sorted(id_to_target.keys())]
elif "target_to_id" in model_meta:
    inv = {int(v): str(k) for k, v in model_meta["target_to_id"].items()}
    class_names = [inv[i] for i in sorted(inv.keys())]
else:
    raise ValueError("No class mapping in model meta")

catboost_mod = importlib.import_module("catboost")
CatBoostClassifier = catboost_mod.CatBoostClassifier
model = CatBoostClassifier()
model.load_model(model_path.as_posix())

for split_df in [train_df, val_df, test_df]:
    for col in ["history_1", "history_2", "history_3", "target"]:
        if col in split_df.columns:
            split_df[col] = split_df[col].astype("string")

print("Loaded split for graph stage:", train_df.shape, val_df.shape, test_df.shape)
print("Graph model key:", GRAPH_MODEL_KEY)
print("Feature columns:", feature_columns)
print("Classes:", class_names)

In [3]:
# 6.1 Build train-only transition graph and evaluate hybrid reranking
transition_counts = (
    train_df.groupby(["history_3", "target"], dropna=False)
    .size()
    .rename("count")
    .reset_index()
    .dropna(subset=["history_3", "target"])
    .astype({"history_3": "string", "target": "string"})
)

transition_prob_matrix = (
    transition_counts.pivot(index="history_3", columns="target", values="count")
    .fillna(0.0)
    .reindex(index=class_names, columns=class_names, fill_value=0.0)
)
transition_prob_matrix = transition_prob_matrix.div(
    transition_prob_matrix.sum(axis=1).replace(0, np.nan), axis=0
).fillna(0.0)

X_val = val_df[feature_columns]
X_test = test_df[feature_columns]
val_proba = np.asarray(model.predict_proba(X_val))
test_proba = np.asarray(model.predict_proba(X_test))

class_to_idx = {name: idx for idx, name in enumerate(class_names)}
trans_values = transition_prob_matrix.to_numpy(dtype=float)

def graph_scores_from_history(split_df):
    hist_idx = split_df["history_3"].astype("string").map(class_to_idx).to_numpy()
    scores = np.zeros((len(split_df), len(class_names)), dtype=float)
    valid_mask = ~pd.isna(hist_idx)
    if valid_mask.any():
        scores[valid_mask] = trans_values[hist_idx[valid_mask].astype(int)]
    return scores

def hybrid_predict_from_proba(split_df, proba, alpha, beta):
    g_scores = graph_scores_from_history(split_df)
    final_scores = alpha * proba + beta * g_scores
    pred_idx = np.argmax(final_scores, axis=1)
    pred_labels = np.asarray(class_names, dtype=object)[pred_idx]
    return pred_labels, final_scores

def metric_row(y_true, y_pred, model_name, split_name, alpha=None, beta=None):
    return {
        "model": model_name,
        "split": split_name,
        "alpha": np.nan if alpha is None else float(alpha),
        "beta": np.nan if beta is None else float(beta),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", labels=class_names, zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", labels=class_names, zero_division=0)),
    }

y_val_true = val_df["target"].astype("string")
y_test_true = test_df["target"].astype("string")

val_pred_cat_idx = np.argmax(val_proba, axis=1)
test_pred_cat_idx = np.argmax(test_proba, axis=1)
val_pred_cat = np.asarray(class_names, dtype=object)[val_pred_cat_idx]
test_pred_cat = np.asarray(class_names, dtype=object)[test_pred_cat_idx]

results_rows = [
    metric_row(y_val_true, val_pred_cat, "catboost", "validation"),
    metric_row(y_test_true, test_pred_cat, "catboost", "test"),
]

alpha_beta_grid = [(0.9, 0.1), (0.8, 0.2), (0.7, 0.3)]
hybrid_store = {}

for alpha, beta in alpha_beta_grid:
    val_pred_h, val_scores = hybrid_predict_from_proba(val_df, val_proba, alpha, beta)
    test_pred_h, test_scores = hybrid_predict_from_proba(test_df, test_proba, alpha, beta)
    hybrid_store[(alpha, beta)] = {
        "val_pred": val_pred_h,
        "test_pred": test_pred_h,
        "val_scores": val_scores,
        "test_scores": test_scores,
    }
    results_rows.append(metric_row(y_val_true, val_pred_h, "hybrid_graph", "validation", alpha, beta))
    results_rows.append(metric_row(y_test_true, test_pred_h, "hybrid_graph", "test", alpha, beta))

results_hybrid_df = pd.DataFrame(results_rows).sort_values(["split", "macro_f1", "accuracy"], ascending=[True, False, False]).reset_index(drop=True)
print("Comparison table: CatBoost baseline vs Hybrid")
display(results_hybrid_df)

hybrid_val = results_hybrid_df[(results_hybrid_df["model"] == "hybrid_graph") & (results_hybrid_df["split"] == "validation")]
best_row = hybrid_val.sort_values(["macro_f1", "accuracy"], ascending=[False, False]).iloc[0]
best_alpha = float(best_row["alpha"])
best_beta = float(best_row["beta"])
best_pair = (best_alpha, best_beta)
best_val_scores = hybrid_store[best_pair]["val_scores"]
best_test_scores = hybrid_store[best_pair]["test_scores"]

def topk_labels(score_matrix, k=3):
    k = min(int(k), score_matrix.shape[1])
    idx_unsorted = np.argpartition(score_matrix, -k, axis=1)[:, -k:]
    scores_unsorted = np.take_along_axis(score_matrix, idx_unsorted, axis=1)
    order = np.argsort(-scores_unsorted, axis=1)
    idx_sorted = np.take_along_axis(idx_unsorted, order, axis=1)
    return np.asarray(class_names, dtype=object)[idx_sorted]

val_top3 = topk_labels(best_val_scores, k=3)
test_top3 = topk_labels(best_test_scores, k=3)
val_hit3 = float(np.mean([y_val_true.iloc[i] in val_top3[i] for i in range(len(y_val_true))]))
test_hit3 = float(np.mean([y_test_true.iloc[i] in test_top3[i] for i in range(len(y_test_true))]))

topk_summary_df = pd.DataFrame([
    {"model": "hybrid_graph", "split": "validation", "alpha": best_alpha, "beta": best_beta, "hit@3": val_hit3},
    {"model": "hybrid_graph", "split": "test", "alpha": best_alpha, "beta": best_beta, "hit@3": test_hit3},
])
print("Top-3 hit@3 summary (best hybrid config):")
display(topk_summary_df)

results_dir = ROOT / "artifacts" / "results" / "graph_hybrid"
results_dir.mkdir(parents=True, exist_ok=True)
results_hybrid_df.to_csv(results_dir / "hybrid_metrics.csv", index=False)
topk_summary_df.to_csv(results_dir / "hybrid_topk_summary.csv", index=False)
transition_prob_matrix.to_pickle(results_dir / "transition_prob_matrix.pkl")
with open(results_dir / "best_hybrid_config.json", "w", encoding="utf-8") as f:
    json.dump({"alpha": best_alpha, "beta": best_beta, "graph_model_key": GRAPH_MODEL_KEY}, f, ensure_ascii=False, indent=2)

print("Saved graph/hybrid artifacts to:", results_dir)

Comparison table: CatBoost baseline vs Hybrid


,model,split,alpha,beta,accuracy,macro_f1,weighted_f1
0,catboost,test,NaN,NaN,0.699006,0.568485,0.689626
1,hybrid_graph,test,0.9,0.1,0.698736,0.564712,0.688797
2,hybrid_graph,test,0.8,0.2,0.697812,0.560300,0.687307
3,hybrid_graph,test,0.7,0.3,0.695507,0.554390,0.684406
4,catboost,validation,NaN,NaN,0.699281,0.568771,0.689966
5,hybrid_graph,validation,0.9,0.1,0.698920,0.564917,0.689056
6,hybrid_graph,validation,0.8,0.2,0.697892,0.560290,0.687448
7,hybrid_graph,validation,0.7,0.3,0.695626,0.554387,0.684578


Top-3 hit@3 summary (best hybrid config):


,model,split,alpha,beta,hit@3
0,hybrid_graph,validation,0.9,0.1,0.949587
1,hybrid_graph,test,0.9,0.1,0.949667


Saved graph/hybrid artifacts to: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\results\graph_hybrid
